

pour les facteurs invariants je l'avais coder au départ pour A non nécessaraiement carré 
et comme dans le cours on considère que des matrices carrés pour les invaraitns de similitude il y a sans doute des points non nécessaire dans mon code

J'ai notamment pu remarquer qu'on calcule les invariants de similitude seulement pour des corps et pas des anneaux :) 

les problèmes d'arrondis ont été fait

In [41]:
# def sage
R.<X>=PolynomialRing(QQ)
P= X**2 + 0*X**3
P.degree()


# pour l'erruer sur les arrondis 
eps= 10e-20
print(eps)

1.00000000000000e-19


In [ ]:

#print(matrice_test)
def find_min(matrice,i): # ok marche
    n=matrice.nrows()
    m=matrice.ncols()
    k,l=-1,-1
    for ii in range(i,n):
        for j in range(i,m):
            if (abs(matrice[ii][j].degree())<abs(matrice[k][l].degree()) or k==-1)and (matrice[ii][j].degree()>0 or abs(int(matrice[ii][j]))>eps):
                k,l=ii,j 
    #if matrice[k][l]==0:
    if (matrice[k][l].degree()==0 and abs(int(matrice[k][l]))<=eps) or (matrice[k][l].degree()>0 and matrice[k][l]==0): # test si P=0
        return -2,-2
    return k,l

In [43]:
def switch_ligne(matrice,i,j):
    matrice.swap_rows(i,j)

def switch_colonne(matrice,i,j):
    matrice.swap_columns(i,j)

In [44]:
def operation_ligne(matrice,i,j,facteur):
    n=matrice.nrows()
    m=matrice.ncols()
    for k in range(m):
        matrice[i,k]= matrice[i,k] +matrice[j,k]*facteur 

def operation_colonne(matrice,i,j,facteur):
    n=matrice.nrows()
    m=matrice.ncols()
    for k in range(n):
        matrice[k,i]= matrice[k,i] +matrice[k,j]*facteur 


In [45]:
def Div_eucl_colonne(matrice,i): # i est le numéro de l'étape # 
    n=matrice.nrows()
    m=matrice.ncols()
    operation=[0 for i in range(n)]
    min=matrice[i][i]
    for k in range(i+1,n): # on obtient les opérations faites sur la premières colonne
        q,r = matrice[k,i].quo_rem(min)
        operation[k] = q
        matrice[k,i]=r
    # on les fait sur le reste des colonnes
    for j in range(i+1,m):
        v=matrice[i][j]
        for k in range(i+1,n):
            matrice[k,j]=matrice[k,j] - operation[k]*v
    return operation 

def Div_eucl_ligne(matrice,i):# i est le numéro de l'étape
    n=matrice.nrows()
    m=matrice.ncols()
    operation=[0 for i in range(m)]
    min=matrice[i][i]
    for k in range(i+1,m):
        q,r = matrice[i,k].quo_rem(min)
        operation[k] = q
        matrice[i,k]=r
        
    for j in range(i+1,n):
        v=matrice[j][i]
        for k in range(i+1,m):
            matrice[j,k]=matrice[j,k] - operation[k]*v
    return operation 

In [ ]:
def test_fini_colonne_k(matrice,k):
    n=matrice.nrows()
    m=matrice.ncols()
    for i in range(k+1,n):
        if matrice[i][k].degree()>0 or abs(int(matrice[i][k]))>eps:
            return False  
    return True
#print(test_fini_colonne_k([[4,6],[2,2]],0))

def test_fini_ligne_k(matrice,k):
    n=matrice.nrows()
    m=matrice.ncols()
    for j in range(k+1,m):
        if matrice[k][j].degree()>0 or abs(int(matrice[k][j]))>eps:
            return False    
    return True

def test_fini_stage_k(matrice,k):
    return test_fini_colonne_k(matrice,k) and test_fini_ligne_k(matrice,k)

In [47]:

def test_divisibilité(matrice,i):
    n=matrice.nrows()
    m=matrice.ncols()
    for j in range(i+1,n):
        for k in range(i+1,m):
            q,r = matrice[j][k].quo_rem(matrice[i][i])
            #if r!=0:
            if r.degree()>0 or abs(int(r))>eps:
                return j,k 
    return -1,-1

In [ ]:
def algo_thmstructure(matrice):
    n=matrice.nrows()
    m=matrice.ncols()
    matrice = X*identity_matrix(R, n) - matrice
    print(matrice)
    P=[[0 for i in range(m)] for j in range(n)] # matrice p des opérations sur les lignes , pas fait au final mais on pas long à rajouter
    p=min(n,m)
    c=100
    i=0
    while i<p and i<c:
        print(matrice)
        print(i)
        b=True# test pour savoir s'il les coefs sont nuls sauf celui sur la diagonale
        j=0
        while b and j<c:
            print(matrice)
            print("op col")
            print(i)
            k,l=find_min(matrice,i) #minimum dans la matrice réduite j,k>=i
            print(k,l)
            if (k,l)==(-2,-2):
                b=False 
                break
            switch_ligne(matrice,i,k)
            switch_colonne(matrice,i,l)
            Div_eucl_colonne(matrice,i)
            if test_fini_colonne_k(matrice,i):
                b=False
            j=j+1
        matrice[i,i]=matrice[i,i] / matrice[i,i].leading_coefficient() # ça peut se faire avec des opéations mais c'est le plus simple
        b=True
        j=0
        while b and j<c:
            print(matrice)
            print("op ligne")
            print(i)
            k,l=find_min(matrice,i) #minimum dans la matrice réduite j,k>=i
            if (k,l)==(-2,-2):
                b=False 
                break
            switch_ligne(matrice,i,k)
            switch_colonne(matrice,i,l)
            Div_eucl_ligne(matrice,i)
            if test_fini_ligne_k(matrice,i):
                b=False
            j=j+1
        matrice[i,i]=matrice[i,i] / matrice[i,i].leading_coefficient() # ça peut se faire avec des opéations mais c'est le plus simple
        # cas d1 ne divise pas d2
        j,k=test_divisibilité(matrice,i)
        if j!=-1: 
            operation_ligne(matrice,i,j,1)
        elif test_fini_stage_k(matrice,i):
            i=i+1
            
            
    return [matrice[i][i] for i in range(p)] # retourne la liste des facteurs invariants de la matrice
matrice_test=[[2,4],[6,8]]
test1=matrice_test
test1=Matrix(QQ, test1)
test2=[[1,0],[1,1]]
test2=Matrix(QQ, test2)
test3=[[4,6],[10,14]] 
test3=Matrix(QQ, test3)
test4=[[2,3],[5,7]] 
test4=Matrix(QQ, test4)
test5=[[2,4],[1,2]]
test5=Matrix(QQ, test5)
test6=[[2,4,6],[4,8,12],[1,2,3]]
test6=Matrix(QQ, test6)
test7=[[3,6,9],[6,15,21],[3,9,12]]
test7=Matrix(QQ, test7)
#print(find_min(test5,0))

A = Matrix(QQ, [[1,2],[3,4]])
test8=[[2,0],[0,3]]
test8=Matrix(QQ, test8)
test9=[[6,0],[0,15]]
test9=Matrix(QQ, test9)
test10=[[4,0,0],[0,6,0],[0,0,9]]
test10=Matrix(QQ, test10)

lambda_=1
N=3
Jordan=[[0 for i in range(N)] for j in range(N)]
Jordan[N-1][N-1]=lambda_
for j in range(N-1):
    Jordan[j][j]=lambda_
    Jordan[j+1][j]=1
Jordan[0][2]+=10**(-8)
Jordan=Matrix(QQ,Jordan)
print(Jordan)
#print(operation_ligne(test8,0,1,1))
print(algo_thmstructure(Jordan)) 

[          1           0 1/100000000]
[          1           1           0]
[          0           1           1]
[       X - 1            0 -1/100000000]
[          -1        X - 1            0]
[           0           -1        X - 1]
[       X - 1            0 -1/100000000]
[          -1        X - 1            0]
[           0           -1        X - 1]
0
[       X - 1            0 -1/100000000]
[          -1        X - 1            0]
[           0           -1        X - 1]
c
0
1 0
[            1         X - 1             0]
[            0 X^2 - 2*X + 1  -1/100000000]
[            0            -1         X - 1]
l
0
[            1             0             0]
[            0 X^2 - 2*X + 1  -1/100000000]
[            0            -1         X - 1]
1
[            1             0             0]
[            0 X^2 - 2*X + 1  -1/100000000]
[            0            -1         X - 1]
c
1
2 1
[                                      1                                       0                 